### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="telemonitoring_parkinsons_biomedical_voice_measurements",
    dataset_year="2007",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C59C74",
    download_description="""
We get the 2009 data from the UCI repository.

wget https://archive.ics.uci.edu/static/public/174/parkinsons.zip && unzip parkinsons.zip telemonitoring/parkinsons_updrs.data && mv telemonitoring/parkinsons_updrs.data parkinsons_updrs.data && rm -rf parkinsons.zip telemonitoring && mkdir -p local-data-warehouse/telemonitoring_parkinsons_biomedical_voice_measurements && mv parkinsons_updrs.data local-data-warehouse/telemonitoring_parkinsons_biomedical_voice_measurements/
""",
    # References
    academic_reference_bibtex="""@article{tsanas2009accurate,
  title={Accurate telemonitoring of Parkinson’s disease progression by non-invasive speech tests},
  author={Tsanas, Athanasios and Little, Max and McSharry, Patrick and Ramig, Lorraine},
  journal={Nature Precedings},
  pages={1--1},
  year={2009},
  publisher={Nature Publishing Group UK London}
}
""",
    academic_reference_bibtex_key="tsanas2009accurate",
    license="CC BY 4.0",
    data_tags=["Non-IID", "Grouped", "WrongDomain"],
    curation_comments="""
We start with the data from UCI.

We aim to simulate the task of predicting the UPDRS score of Parkinson's patients based on their voice measurements over time. This represents the task of only getting the voice measurements to judge the UPDRS. We assume we have no measurement of a patient to make this call. Thus, we have a grouped data task, where we have to holdout entire patients over time. We thus test for one patients multiple time points and repeated measurements at once.

- We use total_UPDRS as target. Note that real UPDRS values were were obtained at baseline, three-month and six-month trial period and all other weekly cases were interpolated. Thus, the ground truth is not perfect and just an estimate based on knowing the future. We reduce it to three values (first, medium <130 days, last), as we have no other ground truth.
- We have several measurements per day for each patient when they were measured. We have some edge cases with negative test time as they got measured before the study started.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="total_UPDRS",
    problem_type="regression",
    objective_metric_name="rmse",
    group_on="subject#",
    group_time_on="test_time",
    group_labels="per_sample",
)

## Preprocessing

In [ ]:
import pandas as pd

df = pd.read_csv(dataset_mold.path / "parkinsons_updrs.data")

df = df.drop(columns=[
    "motor_UPDRS",
])
as_cat_type = ["sex", "subject#"]
df[as_cat_type] = df[as_cat_type].astype("category")

def select_rows(x):
    # Anchor times
    first_time = x.iloc[0]["test_time"]
    last_time = x.iloc[-1]["test_time"]

    before_130 = x[x["test_time"] < 130]
    last_before_130_time = before_130.iloc[-1]["test_time"]

    # Collect times to keep
    times_to_keep = {first_time, last_time}
    times_to_keep.add(last_before_130_time)

    # Keep all rows with matching test_time
    return x[x["test_time"].isin(times_to_keep)]

df = (
    df.groupby("subject#", group_keys=False)
      .apply(select_rows)
      .reset_index(drop=True)
)

df = df.sample(frac=1, random_state=42).sort_values(by=["subject#", "test_time"]).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
    group_labels=task_mold.group_labels,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits = curation_recommendations.get_recommended_grouped_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    group_on=task_mold.group_on,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
    group_labels=task_mold.group_labels,
    show_splits=True,
    target_on=task_mold.target_column_name,
    random_state=split_random_state,
)
splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="We create a default group-label-per-sample split. We thus simulate the use case that someone fits a model and deploys it to predict the UPDRS score of a new patient based on their voice measurements for some time, ignoring the trajectory of the patient itself. We thus have to hold out entire patients and test for one patients multiple time points and repeated measurements at once.",
    splits=splits,
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()